For this we have a more focused look on:
1. DoS / DDoS
2. DoS slowloris (9:47 – 10:10 a.m.)
3. DoS Slowhttptest (10:14 – 10:35 a.m.)
4. DoS Hulk (10:43 – 11 a.m.)
5. DoS GoldenEye (11:10 – 11:23 a.m.)
6. Heartbleed Port 444 (15:12 - 15:32)

Compared to Friday DDoS attacks, here i decided to use SMOTE to keep a certain flow and notice a difference,

In [1]:
#Imports
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [2]:
print("Loading the Wednesday (DoS/DDoS) Dataset")
file_path = "MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv"
df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()

Loading the Wednesday (DoS/DDoS) Dataset


In [3]:
print("\n Traffic Distribution (Before Cleaning)")
#BENIGN, DoS Hulk, DoS GoldenEye, DoS slowloris, DoS Slowhttptest, Heartbleed
print(df['Label'].value_counts())


 Traffic Distribution (Before Cleaning)
Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64


In [4]:
print("\n Cleaning and Encoding Data")
df_clean = df.dropna()
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

df_clean['Label'] = df_clean['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)
print(df_clean['Label'].value_counts())


 Cleaning and Encoding Data
Label
0    439683
1    251723
Name: count, dtype: int64


In [5]:
print("\n Splitting Data (Split BEFORE SMOTE)")
X = df_clean.drop('Label', axis=1)
y = df_clean['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


 Splitting Data (Split BEFORE SMOTE)


In [6]:
print("\n Applying SMOTE to balance the Training Data")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


 Applying SMOTE to balance the Training Data


In [7]:
print("\n Scaling the Data")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)


 Scaling the Data


In [8]:
print("\n Multi-Model Cycling")
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(50,), max_iter=100, random_state=42)
}

results = []


 Multi-Model Cycling


In [9]:
for name, model in models.items():
    print(f"  -> Training {name}...")
    start_time = time.time()
    
    model.fit(X_train_scaled, y_train_smote)
    y_pred = model.predict(X_test_scaled)
    
    train_time = time.time() - start_time
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 4),
        "Recall (Attack Catch Rate)": round(recall * 100, 4),
        "F1-Score": round(f1, 4),
        "Training Time (s)": round(train_time, 2)
    })

  -> Training Decision Tree...
  -> Training Random Forest...
  -> Training XGBoost...
  -> Training LightGBM...


c:\Users\Magda\Desktop\AnomalyDetection\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  -> Training Logistic Regression...
  -> Training MLP (Neural Net)...


In [10]:
print("\n Final Wednesday (DoS/DDoS) Leaderboard")
results_df = pd.DataFrame(results).sort_values(by="Recall (Attack Catch Rate)", ascending=False)
display(results_df)


 Final Wednesday (DoS/DDoS) Leaderboard


,Model,Accuracy (%),Recall (Attack Catch Rate),F1-Score,Training Time (s)
2,XGBoost,99.9812,99.9960,0.9997,21.73
3,LightGBM,99.9754,99.9940,0.9997,23.50
1,Random Forest,99.9559,99.9702,0.9994,66.33
5,MLP (Neural Net),99.7209,99.9643,0.9962,786.05
0,Decision Tree,99.9544,99.9544,0.9994,79.08
4,Logistic Regression,98.5696,98.6607,0.9805,97.78
